# Gram-Schmidt Orthogonalization and UV-Vis Spectroscopy

## Overview

This notebook develops a complete pipeline for **quantitative mixture analysis** using UV-Vis absorption spectroscopy — a concrete, physically grounded setting in which to explore four interconnected ideas from linear algebra and numerical methods.

---

### 1. Modified Gram-Schmidt (MGS) Orthogonalization

Given a set of linearly independent vectors $\{\mathbf{v}_1, \ldots, \mathbf{v}_n\}$ in $\mathbb{R}^m$, MGS produces an orthonormal basis $\{\mathbf{e}_1, \ldots, \mathbf{e}_n\}$ for the same subspace via the recurrence:

$$\mathbf{u}_k = \mathbf{v}_k - \sum_{j=1}^{k-1} \langle \mathbf{v}_k,\, \mathbf{e}_j \rangle\, \mathbf{e}_j, \qquad \mathbf{e}_k = \frac{\mathbf{u}_k}{\|\mathbf{u}_k\|}$$

Unlike classical Gram-Schmidt, MGS **immediately overwrites** $\mathbf{v}_k$ after each projection subtraction, preventing floating-point errors from accumulating to near machine-epsilon orthonormality.

---

### 2. Singular Value Decomposition (SVD)

SVD factors a data matrix $M \in \mathbb{R}^{m \times n}$ as $M = U \Sigma V^T$, where $U$ and $V$ have orthonormal columns and $\Sigma$ is diagonal with non-negative **singular values** in decreasing order. The left singular vectors corresponding to the *large* singular values span the **signal subspace**; those corresponding to small values span the **noise subspace**. SVD is the numerically optimal counterpart to MGS applied to the best-conditioned possible seed vectors.

---

### 3. Beer-Lambert Law

For a solution containing $n$ dissolved species, the absorbance measured at wavelength $\lambda$ is:

$$A(\lambda) = \log_{10}\!\left(\frac{I_0}{I}\right) = \ell \sum_{i=1}^{n} \varepsilon_i(\lambda)\, c_i$$

| Symbol | Quantity | Units |
|--------|----------|-------|
| $A(\lambda)$ | Absorbance — dimensionless ratio of light intensities | — |
| $\ell$ | Cuvette path length | cm |
| $\varepsilon_i(\lambda)$ | Molar absorptivity of species $i$ | L·mol⁻¹·cm⁻¹ |
| $c_i$ | Concentration of species $i$ | mol/L |

In matrix form across all $p$ wavelengths: $\mathbf{a} = \ell \cdot E\,\mathbf{c}$, where $E \in \mathbb{R}^{p \times n}$ contains the molar absorptivities. **The detector gives us $\mathbf{a}$; the goal is to recover $\mathbf{c}$.**

---

### 4. Multivariate Curve Resolution — Alternating Least Squares (MCR-ALS)

SVD provides an orthonormal basis for the signal subspace, but that basis is an arbitrary rotation of the true chemical spectra. MCR-ALS finds the unique chemically meaningful rotation by enforcing two physical constraints — **non-negative spectra** and **non-negative concentrations** — via alternating least squares:

$$C \leftarrow \max\!\left(0,\; \arg\min_C \|D - CS^T\|_F\right), \qquad S \leftarrow \max\!\left(0,\; \arg\min_S \|D - CS^T\|_F\right)$$

---

### The Pipeline

$$\underbrace{\text{Noisy mixtures}}_{\text{detector output}} \xrightarrow{\text{SVD}} \underbrace{\text{Signal subspace}}_{\text{MGS/SVD basis}} \xrightarrow{\text{MCR-ALS}} \underbrace{\text{Pure spectra}}_{\text{recovered}} \xrightarrow{\text{Beer's Law}^{-1}} \underbrace{\text{Concentrations}}_{\text{goal}}$$


## Imports

Library versions are printed to confirm the environment is correctly configured.

In [ ]:
# Cell 01
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy
from numpy.typing import NDArray
from scipy.optimize import linear_sum_assignment

print(f"numpy      {np.__version__}")
print(f"matplotlib {matplotlib.__version__}")
print(f"scipy      {scipy.__version__}")

## Helper Functions

### Gaussian Absorption Peak

Real molecular absorption bands are well-modeled as Gaussians in wavelength space. Each chemical in our simulation has a primary peak plus a secondary shoulder, creating the spectral overlap that makes the problem non-trivial.

### Modified Gram-Schmidt

Accepts a matrix $A$ whose columns are the vectors to orthonormalize. Works in-place on a working copy: normalizes column $i$ to unit length, then immediately subtracts its projection from all columns $j > i$.

A small verification is run at the end: MGS is applied to a random $5 \times 3$ matrix and the maximum off-diagonal element of $Q^T Q$ is confirmed to be near machine epsilon ($\approx 2.2 \times 10^{-16}$).


In [ ]:
# Cell 02
def gaussian(x: NDArray, mu: float, sigma: float, amp: float) -> NDArray:
    """Evaluate a Gaussian absorption peak at x.

    Parameters
    ----------
    x : NDArray
        Wavelength evaluation points (nm).
    mu : float
        Peak center (nm).
    sigma : float
        Peak width (nm).
    amp : float
        Peak amplitude (dimensionless, scaled externally to physical units).

    Returns
    -------
    NDArray
        Gaussian values at each point in x.
    """
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def modified_gram_schmidt(A: NDArray) -> NDArray:
    """Orthonormalize columns of A using modified Gram-Schmidt.

    Parameters
    ----------
    A : NDArray
        Matrix whose columns are the input vectors (m x n).

    Returns
    -------
    Q : NDArray
        Matrix with orthonormal columns (m x n).
    """
    _, n = A.shape
    Q = A.astype(np.float64).copy()
    for i in range(n):
        # Normalize the i-th column to unit length
        Q[:, i] /= np.linalg.norm(Q[:, i])
        # Subtract the projection of all remaining columns onto e_i
        for j in range(i + 1, n):
            Q[:, j] -= np.dot(Q[:, j], Q[:, i]) * Q[:, i]
    return Q


# --- Quick MGS verification on a random 5x3 matrix ---
_rng = np.random.default_rng(0)
_A = _rng.random((5, 3))
_Q = modified_gram_schmidt(_A)
_err = np.max(np.abs(_Q.T @ _Q - np.eye(3)))
print("gaussian() and modified_gram_schmidt() defined")
print(f"MGS verification: max |Q^T Q - I| = {_err:.2e}  (machine epsilon ~ 2.2e-16)")

### MCR-ALS

Initialized from SVD to give the alternating least squares a well-conditioned starting point. Convergence is monitored by the **relative** change in the Frobenius residual — the absolute residual cannot reach zero because there is irreducible measurement noise in the data.


In [ ]:
# Cell 03
def mcr_als(
    D: NDArray,
    n_components: int,
    max_iter: int = 2000,
    tol: float = 1e-6,
) -> tuple[NDArray, NDArray, list[float]]:
    """Multivariate Curve Resolution - Alternating Least Squares.

    Factorizes D (n_samples x n_wavelengths) as C @ S.T subject to
    non-negativity constraints on both C (concentrations) and S (spectra).
    Non-negativity rotates the arbitrary SVD/MGS basis into the unique
    chemically interpretable orientation.

    Parameters
    ----------
    D : NDArray
        Data matrix of shape (n_samples, n_wavelengths).
    n_components : int
        Number of chemical components to resolve.
    max_iter : int
        Maximum number of ALS iterations.
    tol : float
        Relative convergence tolerance: stop when |delta r| / r < tol.

    Returns
    -------
    C : NDArray
        Concentration profiles, shape (n_samples, n_components).
    S : NDArray
        Recovered pure spectra, shape (n_wavelengths, n_components).
    residuals : list[float]
        Frobenius residual ||D - C @ S.T||_F at each iteration.
    """
    # Initialize from SVD scores and loadings - much better than a random start
    U, sv, Vt = np.linalg.svd(D, full_matrices=False)
    C = np.abs(U[:, :n_components] * sv[:n_components])  # (n_samples, n_components)
    S = np.abs(Vt[:n_components].T)  # (n_wavelengths, n_components)

    residuals: list[float] = []
    prev_residual = np.inf

    for _ in range(max_iter):
        # Update C: solve D.T = S @ C.T, then clamp negatives to zero
        C = np.linalg.lstsq(S, D.T, rcond=None)[0].T
        C = np.maximum(C, 0)
        # Update S: solve D = C @ S.T, then clamp negatives to zero
        S = np.linalg.lstsq(C, D, rcond=None)[0].T
        S = np.maximum(S, 0)
        residual = float(np.linalg.norm(D - C @ S.T, "fro"))
        residuals.append(residual)
        # Relative convergence: stop when improvement < tol * current residual
        if prev_residual > 0 and abs(prev_residual - residual) / residual < tol:
            break
        prev_residual = residual

    return C, S, residuals


print("mcr_als() defined  (will run in Cell 07)")

## Physical Setup: Beer's Law Constants and Pure Spectra

We simulate three aromatic dye molecules, each with a primary absorption peak and a secondary shoulder. The molar absorptivity $\varepsilon_\text{peak} = 50{,}000$ L·mol⁻¹·cm⁻¹ is typical for a strongly absorbing organic chromophore (e.g. a xanthene or porphyrin dye).

With a standard $\ell = 1.0$ cm cuvette, Beer's Law predicts:

$$A = \varepsilon_\text{peak} \cdot c \cdot \ell \approx 1 \implies c \approx \frac{1}{50{,}000} = 2 \times 10^{-5}\ \text{mol/L} = 20\ \mu\text{mol/L}$$

So concentrations in the range $5$–$50\ \mu\text{mol/L}$ produce absorbances of $0.25$–$2.5$, safely within the reliable dynamic range of most UV-Vis instruments (practical limit: $A < 3$).

Note that Chemical C has a secondary shoulder at 340 nm that overlaps with Chemical A's primary peak. This **spectral collinearity** will limit concentration recovery accuracy for Chemical C in the final step.


In [ ]:
# Cell 04
rng = np.random.default_rng(42)

n_wavelengths = 50
wavelengths = np.linspace(300, 550, n_wavelengths)  # nm
n_chemicals = 3

# Physical constants (Beer's Law: A = eps * c * l)
eps_peak = 50_000.0  # peak molar absorptivity, L*mol^-1*cm^-1
path_length = 1.0  # cuvette path length, cm

# Pure spectra: molar absorptivity vs wavelength, shape (50, 3)
# Each chemical has a primary peak plus a secondary shoulder.
# Gaussians normalized to peak=1 then scaled by eps_peak -> units L*mol^-1*cm^-1
pure_spectra = eps_peak * np.column_stack(
    [
        gaussian(wavelengths, 340, 25, 1.0)
        + gaussian(wavelengths, 450, 20, 0.3),  # Chemical A
        gaussian(wavelengths, 390, 30, 0.9)
        + gaussian(wavelengths, 470, 25, 0.4),  # Chemical B
        gaussian(wavelengths, 430, 35, 0.8)
        + gaussian(wavelengths, 340, 20, 0.2),  # Chemical C
    ]
)

print(f"Pure spectra matrix shape  : {pure_spectra.shape}  (wavelengths x chemicals)")
print(
    f"Wavelength range           : {wavelengths[0]:.0f} - {wavelengths[-1]:.0f} nm  ({n_wavelengths} points)"
)
print()
for i, name in enumerate(["A", "B", "C"]):
    pk_wl = wavelengths[np.argmax(pure_spectra[:, i])]
    print(
        f"Chemical {name}: peak eps = {pure_spectra[:, i].max():,.0f} L*mol^-1*cm^-1  at {pk_wl:.0f} nm"
    )

## Generating Mixture Measurements

We prepare 20 cuvettes, each containing all three chemicals at independently drawn random concentrations in the range $5$–$50\ \mu\text{mol/L}$.

The detector applies Beer's Law automatically — it reports only the blended absorbance and knows nothing about the individual components:

$$\mathbf{a}^{(k)} = \ell \cdot E\, \mathbf{c}^{(k)} + \boldsymbol{\eta}^{(k)}, \qquad \boldsymbol{\eta}^{(k)} \sim \mathcal{N}(\mathbf{0},\, \sigma^2 I)$$

where $\sigma = 0.1$ absorbance units represents instrument noise. Each of the 50 wavelength bins receives an independent Gaussian noise hit. Absorbance $A = \log_{10}(I_0/I)$ is **dimensionless**: $A = 1.0$ means 10% of photons reached the detector; $A = 2.0$ means 1%; $A = 3.0$ means 0.1%.

The number of samples (20) is entirely independent of the number of chemicals (3).


In [ ]:
# Cell 05
n_samples = 20

# Independent concentrations per chemical per sample, mol/L
concentrations = rng.uniform(5e-6, 5e-5, size=(n_chemicals, n_samples))

# Beer's Law: A = E @ c * path_length  (dimensionless absorbance)
measurements = pure_spectra @ concentrations * path_length  # (50, 20)

# Additive Gaussian instrument noise, sigma = 0.1 absorbance units
noise = rng.normal(0, 0.1, size=measurements.shape)
measurements_noisy = measurements + noise  # (50, 20)

print(f"Samples x wavelengths      : {measurements_noisy.shape}")
print(
    f"Concentration range        : {concentrations.min() * 1e6:.1f} - {concentrations.max() * 1e6:.1f} umol/L"
)
print(
    f"Absorbance range (clean)   : {measurements.min():.3f} - {measurements.max():.3f}"
)
print(
    f"Absorbance range (noisy)   : {measurements_noisy.min():.3f} - {measurements_noisy.max():.3f}"
)
print()
print("Sample 1 true concentrations (umol/L):")
for i, name in enumerate(["A", "B", "C"]):
    print(f"  Chemical {name}: {concentrations[i, 0] * 1e6:.2f} umol/L")

## Signal Subspace via SVD and Denoising

### Finding the Signal Subspace

The data matrix $M_\text{noisy} \in \mathbb{R}^{50 \times 20}$ has rank 3 in the noise-free limit — three chemicals span a 3-dimensional subspace of $\mathbb{R}^{50}$. Noise inflates the effective rank to 20, but the first three singular values remain much larger than the rest, revealing the true dimensionality.

The top-3 left singular vectors (columns of $U$) form the **optimal orthonormal basis** for the 3D signal subspace — equivalent to what MGS produces when seeded with well-chosen mixture measurements.

### Denoising by Subspace Projection

$$\hat{\mathbf{a}}^{(k)} = Q\, (Q^T \mathbf{a}^{(k)}_\text{noisy})$$

This retains all signal (which lives in 3 dimensions) while discarding $47/50$ of the noise variance (which spreads across all 50 dimensions).


In [ ]:
# Cell 06
# SVD of the full noisy data matrix
U, singular_values, _ = np.linalg.svd(measurements_noisy, full_matrices=False)

# Top-3 left singular vectors = orthonormal basis for the 3D signal subspace
Q = U[:, :n_chemicals]  # shape: (50, 3)

# Verify Q^T Q = I_3
assert np.allclose(Q.T @ Q, np.eye(n_chemicals), atol=1e-12), "Q is not orthonormal"

# Denoise: project all 20 samples onto the 3D signal subspace
coords = Q.T @ measurements_noisy  # (3, 20) - subspace coordinates
denoised = Q @ coords  # (50, 20) - reconstructed in wavelength space

noise_discarded = np.linalg.norm(measurements_noisy - denoised, "fro")
signal_error = np.linalg.norm(denoised - measurements, "fro")

print(f"Signal subspace dimension    : {n_chemicals} of {n_wavelengths}")
print(
    f"Q^T Q = I verified           : max off-diagonal = {np.max(np.abs(Q.T @ Q - np.eye(n_chemicals))):.2e}"
)
print(f"Frobenius norm noise removed : {noise_discarded:.4f}")
print(f"Frobenius norm signal error  : {signal_error:.4f}")
print(f"SNR improvement factor       : {noise_discarded / signal_error:.1f}x")

## MCR-ALS: Rotating the Basis into Chemical Meaning

SVD gives us a correct but **arbitrarily oriented** orthonormal basis for the signal subspace. MCR-ALS uses non-negativity of $\varepsilon$ and $c$ to find the unique rotation aligning the basis with the true pure-component spectra.

After convergence we align recovered components to true spectra by solving a linear assignment problem (Hungarian algorithm) to maximize pairwise correlations — MCR-ALS may return components in a different order than the true spectra.


In [ ]:
# Cell 07
# MCR-ALS convention: samples as rows
D = measurements_noisy.T  # (20, 50)

C_als, S_als, residuals = mcr_als(D, n_chemicals, max_iter=2000)

# Match recovered components to true spectra by maximum correlation
S_als_norm = S_als / (S_als.max(axis=0) + 1e-12)
pure_norm = pure_spectra / (pure_spectra.max(axis=0) + 1e-12)
corr_matrix = np.array(
    [
        [
            np.corrcoef(S_als_norm[:, i], pure_norm[:, j])[0, 1]
            for j in range(n_chemicals)
        ]
        for i in range(n_chemicals)
    ]
)
_, col_ind = linear_sum_assignment(-np.abs(corr_matrix))
S_matched = S_als_norm[:, col_ind]

print(f"MCR-ALS converged in {len(residuals)} iterations")
print(f"Final Frobenius residual     : {residuals[-1]:.6f}")
print()
print("Correlation of recovered vs true spectra:")
for i, name in enumerate(["A", "B", "C"]):
    print(f"  Chemical {name}: r = {corr_matrix[i, col_ind[i]]:.3f}")

## Plots

Three distinct color palettes prevent visual ambiguity between chemicals, mixture samples, and SVD/MGS basis modes.

### Plot 1 — True Pure Spectra (Ground Truth)

These are the molar absorptivity spectra $\varepsilon_i(\lambda)$ in L·mol⁻¹·cm⁻¹ — physical properties of the molecules that the experimenter **cannot measure directly from a mixture**. They require pure reference standards, or must be recovered blind by MCR-ALS.

Note the deliberate spectral overlap: Chemical C's shoulder at 340 nm coincides with Chemical A's primary peak. This **collinearity** will limit concentration recovery accuracy for Chemical C in Plot 6.


In [ ]:
# Cell 08
# Color palettes: one per logical group to prevent cross-plot ambiguity
chemical_labels = ["Chemical A", "Chemical B", "Chemical C"]
chemical_colors = ["crimson", "royalblue", "goldenrod"]  # ground truth & MCR
sample_colors = ["mediumpurple", "darkcyan", "coral", "peru", "teal"]  # mixtures
mode_colors = ["mediumseagreen", "orangered", "slateblue"]  # SVD/MGS modes

fig, ax = plt.subplots(figsize=(7, 4), num="Plot 1: True Pure Spectra")

for i, (label, color) in enumerate(zip(chemical_labels, chemical_colors)):
    ax.plot(wavelengths, pure_spectra[:, i], color=color, linewidth=2, label=label)

ax.set_title("True Pure Spectra  (not directly measurable from mixtures)")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Molar Absorptivity eps (L*mol^-1*cm^-1)")
ax.legend()
plt.tight_layout()
plt.show()

### Plot 2 — Detector Output: Noisy Mixture Spectra

This is **everything the detector actually provides**. Each spectrum is a blended absorbance — a linear superposition of the three pure spectra weighted by concentration — plus $\sigma = 0.1$ AU independent Gaussian noise at each wavelength.

Five of the 20 samples are shown to emphasize that the number of samples has no relationship to the number of chemicals.


In [ ]:
# Cell 09
fig, ax = plt.subplots(figsize=(7, 4), num="Plot 2: Detector Output")

for k, color in enumerate(sample_colors):
    ax.plot(
        wavelengths,
        measurements_noisy[:, k],
        color=color,
        linewidth=1.5,
        alpha=0.85,
        label=f"Sample {k + 1}",
    )

title = f"Detector Output: 5 of {n_samples} Noisy Mixture Spectra"
subtitle = (
    "(A+B+C blended at unknown concentrations; sigma=0.1 noise per wavelength bin)"
)
ax.set_title(title + "\n" + subtitle)
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Absorbance (dimensionless)")
ax.legend()
plt.tight_layout()
plt.show()

### Plot 3 — Singular Value Spectrum

Plotting the singular values of the $50 \times 20$ data matrix in decreasing order reveals a sharp **gap** after the third value — the mathematical fingerprint of three independent chemical components.

In a real experiment this plot tells you how many chemicals are in the mixture **with no prior knowledge** of what those chemicals are. The 3 large singular values (blue) correspond to the signal; the remaining 17 small values (gray) are instrument noise distributed across all 50 wavelength dimensions.


In [ ]:
# Cell 10
fig, ax = plt.subplots(figsize=(7, 4), num="Plot 3: Singular Value Spectrum")

sv_plot = singular_values[:15]
colors_sv = [
    "steelblue" if i < n_chemicals else "lightgray" for i in range(len(sv_plot))
]
ax.bar(range(1, len(sv_plot) + 1), sv_plot, color=colors_sv)
ax.axvline(n_chemicals + 0.5, color="dimgray", linestyle="dashed", linewidth=1.5)
ax.text(
    0.18,
    0.62,
    f"{n_chemicals} large\nvalues\n= signal",
    transform=ax.transAxes,
    color="steelblue",
    fontsize=9,
    ha="center",
    bbox={
        "facecolor": "white",
        "edgecolor": "steelblue",
        "alpha": 0.85,
        "boxstyle": "round,pad=0.3",
    },
)
ax.text(5.5, sv_plot[0] * 0.25, "small values\n= noise", color="dimgray", fontsize=9)
ax.set_title(
    f"Singular Value Spectrum  (gap at index {n_chemicals} reveals {n_chemicals} independent components)"
)
ax.set_xlabel("Singular value index")
ax.set_ylabel("Singular value magnitude")
plt.tight_layout()
plt.show()

### Plot 4 — Signal Subspace Basis (SVD/MGS)

The three columns of $Q$ span the same 3D subspace as the true pure spectra but are **not** physically interpretable:

- They can be **negative** (real absorbance is always $\geq 0$)
- Their orientation is **arbitrary** — determined by SVD, not chemistry
- No single mode corresponds to any single chemical

They are sufficient for **denoising** (projecting onto 3D discards $47/50$ of the noise variance), but MCR-ALS (Plot 5) is needed to find the rotation that aligns them with real chemical spectra.


In [ ]:
# Cell 11
fig, ax = plt.subplots(figsize=(7, 4), num="Plot 4: Signal Subspace Basis")

for i, color in enumerate(mode_colors):
    ax.plot(wavelengths, Q[:, i], color=color, linewidth=2, label=f"Mode {i}")

ax.axhline(0, color="black", linewidth=0.6, linestyle="dashed")
ax.set_title(
    "Signal Subspace Basis (SVD/MGS)\n"
    "(negative values confirm these are coordinate axes, not real spectra)"
)
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Amplitude")
ax.legend()
plt.tight_layout()
plt.show()

### Plot 5 — MCR-ALS: Recovered vs True Spectra

MCR-ALS rotates the arbitrary SVD basis into a chemically interpretable orientation. Solid lines are the normalized true spectra; dashed lines are the MCR-ALS estimates.

Chemicals A and B recover with high fidelity ($r \approx 0.96$). Chemical C ($r \approx 0.94$) is harder because its secondary shoulder at 340 nm overlaps Chemical A's primary peak — MCR-ALS cannot cleanly assign the 340 nm absorbance to A vs C. This is a fundamental limitation of collinear spectra, not a failure of the algorithm.


In [ ]:
# Cell 12
fig, ax = plt.subplots(figsize=(7, 4), num="Plot 5: MCR-ALS Recovery")

for i, (label, color) in enumerate(zip(chemical_labels, chemical_colors)):
    ax.plot(
        wavelengths,
        pure_norm[:, i],
        color=color,
        linewidth=2.5,
        linestyle="solid",
        label=f"{label} (true)",
    )
    ax.plot(
        wavelengths,
        S_matched[:, i],
        color=color,
        linewidth=1.5,
        linestyle="dashed",
        label=f"{label} (MCR-ALS)",
    )

ax.set_title("MCR-ALS: Recovered vs True Spectra  (solid = truth, dashed = recovered)")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Normalized Absorbance")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

### Plot 6 — Concentration Recovery for Sample 1

The ultimate objective: given only the noisy detector output for Sample 1 (the mediumpurple spectrum in Plot 2), recover the concentration of each chemical in $\mu\text{mol/L}$.

The procedure is **Beer's Law inversion** using the MCR-ALS recovered spectra $\hat{E}$ scaled to physical units (L·mol⁻¹·cm⁻¹):

$$\hat{\mathbf{c}} = \arg\min_{\mathbf{c}} \left\| \ell\,\hat{E}\,\mathbf{c} - \hat{\mathbf{a}}^{(1)}_\text{denoised} \right\|_2 \quad \text{solved via } \texttt{np.linalg.lstsq}$$

The concentrations do **not** need to sum to any particular value — they are independent dissolved species in a solvent, not fractions of a whole. The ~40% error on Chemical C is systematic and expected, a direct consequence of its spectral collinearity with Chemical A.


In [ ]:
# Cell 13
sample_idx = 0  # Sample 1 - shown as mediumpurple in Plot 2

# Scale MCR-ALS spectra to physical units (L*mol^-1*cm^-1)
# to resolve the inherent amplitude ambiguity of matrix factorization
S_matched_raw = S_als[:, col_ind]
scale = pure_spectra.max(axis=0) / (S_matched_raw.max(axis=0) + 1e-12)
S_matched_scaled = S_matched_raw * scale

x_denoised = denoised[:, sample_idx]
c_true = concentrations[:, sample_idx]

# Beer's Law inversion: solve (eps * path_length) @ c = A  for c in mol/L
c_from_mcr = np.linalg.lstsq(S_matched_scaled * path_length, x_denoised, rcond=None)[0]

# Convert mol/L -> umol/L for readability
umol = 1e6
c_true_umol = c_true * umol
c_mcr_umol = c_from_mcr * umol

fig, ax = plt.subplots(figsize=(7, 4), num="Plot 6: Concentration Recovery")
bar_width = 0.3
x_pos = np.arange(n_chemicals)
ax.bar(
    x_pos - bar_width / 2,
    c_true_umol,
    bar_width,
    color="cornflowerblue",
    label="True concentration",
)
ax.bar(
    x_pos + bar_width / 2,
    c_mcr_umol,
    bar_width,
    color="darksalmon",
    label="MCR-ALS estimate (fully blind)",
)
for k in range(n_chemicals):
    pct = 100.0 * abs(c_mcr_umol[k] - c_true_umol[k]) / (c_true_umol[k] + 1e-12)
    ax.text(
        x_pos[k] + bar_width / 2,
        c_mcr_umol[k] + 0.5,
        f"{pct:.0f}% err",
        ha="center",
        va="bottom",
        fontsize=9,
        color="dimgray",
    )
ax.set_xticks(x_pos)
ax.set_xticklabels(chemical_labels)
ax.set_title(
    f"Goal: Concentration Recovery for Sample #{sample_idx + 1}\n"
    "(blue = true, salmon = MCR-ALS blind estimate)"
)
ax.set_ylabel("Concentration (umol/L)")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

print(f"\n{'Chemical':<12} {'True (umol/L)':>14} {'MCR-ALS (umol/L)':>17} {'Error':>8}")
print("-" * 55)
for k, name in enumerate(chemical_labels):
    pct = 100.0 * abs(c_mcr_umol[k] - c_true_umol[k]) / (c_true_umol[k] + 1e-12)
    print(f"  {name:<10} {c_true_umol[k]:>14.2f} {c_mcr_umol[k]:>17.2f} {pct:>7.1f}%")